In [1]:
from torch.utils.data import Dataset, DataLoader
import torch
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans, HDBSCAN
# GaussianMixture
from sklearn.mixture import GaussianMixture

import matplotlib.pyplot as plt
import yfinance as yf
import seaborn as sns
# import
from hmmlearn.hmm import GaussianHMM
import pickle

In [2]:
device ="mps"

In [3]:
with open('../../data/complete_features.pkl', 'rb') as f:
    all_data = pickle.load(f)

In [4]:
data = all_data["scaled_featured"].copy()

In [5]:
data

,rolling_sharpe_20,rolling_max_dd_60,realized_vol_20,vol_slope_20,autocorr_20,rel_spx_20,usdinr_vol_20,rel_gold_20,vix_level_20,vix_slope_20,trend_strength_200,sector_dispersion_ratio_20
Date,,,,,,,,,,,,
2008-07-07,-1.220376,-1.882225,2.081321,-0.864592,-1.135796,-0.628471,-0.569023,-2.119483,1.416877,0.823997,-2.165333,-0.568676
2008-07-08,-1.211754,-1.882225,2.077736,-0.022097,-1.117078,-1.046961,-0.542321,-2.455523,1.450259,0.981827,-2.206775,-0.513460
2008-07-09,-0.957426,-1.882225,2.289506,1.636228,-1.218247,-0.338846,-0.561373,-1.988903,1.476862,0.785786,-1.960992,-0.565134
2008-07-10,-0.975703,-1.882225,2.287995,-0.006124,-1.013921,-0.474801,-0.596656,-2.401313,1.493754,0.505013,-1.940271,-0.562557
2008-07-11,-1.136144,-1.882225,2.391761,0.804554,-1.050675,-0.396400,-0.475448,-2.987661,1.507027,0.400347,-2.078101,-0.510752
...,...,...,...,...,...,...,...,...,...,...,...,...
2025-09-15,0.375855,0.654743,-0.795604,0.021503,0.541634,-0.126723,-0.727148,-1.189892,-0.988672,-0.309058,0.314003,0.507783
2025-09-16,0.280399,0.654743,-0.833663,-0.287568,0.045456,-0.238463,-0.708679,-1.276291,-1.000565,-0.327333,0.453848,0.513354
2025-09-17,0.260114,0.654743,-0.836851,-0.019044,0.154682,-0.363408,-0.703390,-1.335486,-1.009414,-0.239281,0.522324,0.511624


In [6]:
close = all_data["Close"].copy()
close

Date
2008-07-07     4030.000000
2008-07-08     3988.550049
2008-07-09     4157.100098
2008-07-10     4162.200195
2008-07-11     4049.000000
                  ...     
2025-09-15    25069.199219
2025-09-16    25239.099609
2025-09-17    25330.250000
2025-09-18    25423.599609
2025-09-19    25327.050781
Name: Close, Length: 4219, dtype: float64

In [7]:
data.shape

(4219, 12)

In [9]:
class TimeSeriesWindowDataset(Dataset):
    def __init__(self, data, window_size=60):
        """
        data: numpy array [T, D]
        """
        self.data = torch.tensor(data, dtype=torch.float32)
        self.window_size = window_size

    def __len__(self):
        return len(self.data) - self.window_size + 1

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.window_size]
        return x

window_size = 60
val_size = 0.2

train_size = int(len(data) * (1 - val_size))
train_data = data.iloc[:train_size]
val_data = data.iloc[train_size:]

train_dataset = TimeSeriesWindowDataset(train_data.values, window_size)
val_dataset = TimeSeriesWindowDataset(val_data.values, window_size)
combined_dataset = TimeSeriesWindowDataset(data.values, window_size)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
combined_loader = DataLoader(combined_dataset, batch_size=batch_size, shuffle=False)



In [10]:
len(combined_dataset)

4160

In [11]:
for batch in train_loader:
    print(batch.shape)  # Should print torch.Size([32, 60, D])
    break

torch.Size([32, 60, 12])


In [13]:
import torch.nn as nn
import torch.nn.functional as F

class TS2VecModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=5):
        super(TS2VecModel, self).__init__()
        layers = []
        for i in range(num_layers):
            dilation = 2 ** i
            layers.append(nn.Conv1d(input_dim if i == 0 else hidden_dim, hidden_dim, kernel_size=3, padding=dilation, dilation=dilation))
            layers.append(nn.ReLU())
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        # x: [B, T, D]
        x = x.permute(0, 2, 1)  # [B, D, T]
        x = self.network(x)     # [B, H, T]
        x = x.permute(0, 2, 1)  # [B, T, H]
        # normalize
        x = F.normalize(x, p=2, dim=-1)
        return x

In [14]:
data.shape

(4219, 12)

In [ ]:
model = TS2VecModel(input_dim=data.shape[1], hidden_dim=128, num_layers=6).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [156]:
model

TS2VecModel(
  (network): Sequential(
    (0): Conv1d(12, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (1): ReLU()
    (2): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(2,), dilation=(2,))
    (3): ReLU()
    (4): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(4,), dilation=(4,))
    (5): ReLU()
    (6): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(8,), dilation=(8,))
    (7): ReLU()
  )
)

In [157]:
def time_mask(x, mask_ratio=0.2):
    B, T, D = x.shape
    mask_len = int(T * mask_ratio)
    start = np.random.randint(0, T - mask_len)
    x = x.clone()
    x[:, start:start+mask_len, :] = 0
    return x

def temporal_negative_mask(B, T, exclusion_radius, device):
    """
    Returns mask of shape [B*T, B*T]
    True = allowed negative
    False = masked out
    """
    BT = B * T
    mask = torch.ones(BT, BT, device=device, dtype=torch.bool)

    for b in range(B):
        for t in range(T):
            idx = b*T + t
            start = max(0, t - exclusion_radius)
            end   = min(T, t + exclusion_radius + 1)
            for tt in range(start, end):
                mask[idx, b*T + tt] = False

    return mask



def jitter(x, sigma=0.02):
    return x + sigma * torch.randn_like(x)


# def temporal_pooling(z):
#     """
#     z: [B, T, C]
#     returns: [B, T//2, C]
#     """
#     if z.size(1) % 2 == 1:
#         z = z[:, :-1, :]
#     z = z.reshape(z.size(0), z.size(1)//2, 2, z.size(2))
#     return z.max(dim=2).values

def temporal_pooling(z):
    if z.size(1) % 2 == 1:
        z = z[:, :-1]
    return z.reshape(z.size(0), z.size(1)//2, 2, z.size(2)).mean(dim=2)

def instance_contrastive_loss(z1, z2, temperature=0.1):
    """
    z1, z2: [B, T, C]
    """
    B, T, C = z1.shape

    z1 = z1.reshape(B*T, C)
    z2 = z2.reshape(B*T, C)

    sim = torch.matmul(z1, z2.T) / temperature  # [BT, BT]
    labels = torch.arange(B*T, device=device)
    
    return F.cross_entropy(sim, labels)
    
def masked_instance_contrastive_loss(
    z1, z2, exclusion_radius=5, temperature=0.1
):
    B, T, C = z1.shape
    z1 = z1.reshape(B*T, C)
    z2 = z2.reshape(B*T, C)

    sim = torch.matmul(z1, z2.T) / temperature
    mask = temporal_negative_mask(B, T, exclusion_radius, z1.device)

    # mask invalid negatives
    sim = sim.masked_fill(~mask, -1e9)

    labels = torch.arange(B*T, device=z1.device)
    return F.cross_entropy(sim, labels)

def hierarchical_contrastive_loss(z1, z2, temperature=0.1, min_time=2):
    """
    z1, z2: [B, T, C]
    """
    total_loss = 0.0
    depth = 0

    while z1.size(1) >= min_time:
        loss = instance_contrastive_loss(z1, z2, temperature)
        total_loss += loss

        # downsample
        z1 = temporal_pooling(z1)
        z2 = temporal_pooling(z2)
        depth += 1

    return total_loss / depth

def ts2vec_contrastive_loss(
    z1, z2,
    temperature=0.1,
    exclusion_radius=5
):
    """
    z1, z2: [B, T, C]
    """
    B, T, C = z1.shape
    loss = 0.0
    count = 0

    for t in range(T):
        pos_sim = F.cosine_similarity(
            z1[:, t], z2[:, t], dim=-1
        ) / temperature  # [B]

        # negatives: all other timestamps except nearby
        neg_sims = []
        for tt in range(T):
            if abs(tt - t) > exclusion_radius:
                neg = F.cosine_similarity(
                    z1[:, t].unsqueeze(1),
                    z2[:, tt].unsqueeze(0),
                    dim=-1
                ) / temperature
                neg_sims.append(neg)

        if len(neg_sims) == 0:
            continue

        neg_sims = torch.cat(neg_sims, dim=1)

        logits = torch.cat(
            [pos_sim.unsqueeze(1), neg_sims], dim=1
        )

        labels = torch.zeros(B, dtype=torch.long, device=z1.device)
        loss += F.cross_entropy(logits, labels)
        count += 1

    return loss / count

def ts2vec_contrastive_loss_vectorized(
    z1, z2,
    temperature=0.1,
    base_exclusion_radius=5
):
    """
    z1, z2: [B, T, C]
    """
    B, T, C = z1.shape
    device = z1.device

    # flatten (instance, time)
    z1_flat = z1.reshape(B*T, C)
    z2_flat = z2.reshape(B*T, C)

    # cosine similarity == dot product because embeddings are normalized
    sim = torch.matmul(z1_flat, z2_flat.T) / temperature   # [BT, BT]

    # ----- temporal negative mask -----
    exclusion_radius = min(base_exclusion_radius, (T - 1) // 2)

    if exclusion_radius == 0:
        return torch.tensor(0.0, device=device)

    # time index per row
    time_idx = torch.arange(T, device=device).repeat(B)     # [BT]

    # batch index per row
    batch_idx = torch.arange(B, device=device).repeat_interleave(T)

    # same batch & temporally close → mask out
    temporal_dist = torch.abs(time_idx[:, None] - time_idx[None, :])
    same_batch = batch_idx[:, None] == batch_idx[None, :]

    invalid_negatives = same_batch & (temporal_dist <= exclusion_radius)

    # allow diagonal (positive pairs)
    diag = torch.eye(B*T, device=device, dtype=torch.bool)
    invalid_negatives = invalid_negatives & (~diag)

    # mask invalid negatives
    sim = sim.masked_fill(invalid_negatives, -1e9)

    # positives are diagonal
    labels = torch.arange(B*T, device=device)

    return F.cross_entropy(sim, labels)


def hierarchical_ts2vec_loss(
    z1, z2,
    temperature=0.1,
    exclusion_radius=5,
    min_time=2
):
    total_loss = 0.0
    depth = 0

    while z1.size(1) >= min_time:
        total_loss += ts2vec_contrastive_loss_vectorized(
            z1, z2,
            temperature,
            exclusion_radius
        )

        z1 = temporal_pooling(z1)
        z2 = temporal_pooling(z2)
        depth += 1

    return total_loss / depth

def hierarchical_ts2vec_loss_v2(
    z1, z2,
    temperature=0.1,
    exclusion_radius=5,
    min_time=2
):
    total_loss = 0.0
    depth = 0

    while z1.size(1) >= min_time:
        T = z1.size(1)

        if T > 2 * exclusion_radius + 1:
            total_loss += ts2vec_contrastive_loss_vectorized(
                z1, z2,
                temperature,
                exclusion_radius
            )
            depth += 1

        z1 = temporal_pooling(z1)
        z2 = temporal_pooling(z2)

    return total_loss / max(depth, 1)



def embedding_variance(z):
    # z: [B, T, C]
    return z.var(dim=0).mean().item() # mean over T and C


In [158]:
%%time
# embedding_variance_1 = []
# embedding_variance_2 = []
for epoch in range(10):
    total_loss = 0
    model.train()
    for x in train_loader:
        x=x.to(device)
        x1 = jitter(time_mask(x))
        x2 = jitter(time_mask(x))
        z1 = model(x1)
        z2 = model(x2)

        # embedding_variance_1.append(embedding_variance(z1))
        # embedding_variance_2.append(embedding_variance(z2))

        loss = hierarchical_ts2vec_loss_v2(z1, z2)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        # print(f"Batch Loss: {loss.item():.4f}")
    # model.eval()
    # with torch.no_grad():
    #     val_loss = 0
    #     for x in val_loader:
    #         x = x.to(device)
    #         x1 = jitter(time_mask(x))
    #         x2 = jitter(time_mask(x))
    #         z1 = model(x1)
    #         z2 = model(x2)
    #         val_loss += hierarchical_ts2vec_loss_v2(z1, z2).item()

    # val_loss /= len(val_loader)

    print(f"Epoch {epoch}, Loss {total_loss / len(train_loader):.4f}")

Epoch 0, Loss 1.6492
Epoch 1, Loss 0.8819
Epoch 2, Loss 0.7300
Epoch 3, Loss 0.6583
Epoch 4, Loss 0.6056
Epoch 5, Loss 0.5761
Epoch 6, Loss 0.5674
Epoch 7, Loss 0.5340
Epoch 8, Loss 0.5264
Epoch 9, Loss 0.5325
CPU times: user 7.21 s, sys: 1.53 s, total: 8.74 s
Wall time: 7.47 s


In [159]:
# save model
# torch.save(model.state_dict(), './../../models/ts2vec_nifty.pth')

In [160]:
# load model
# model.load_state_dict(torch.load('../../models/ts2vec_nifty.pth'))

In [161]:
all_embeddings = []
with torch.inference_mode():
    for x in combined_loader:
        # print("x.shape", x.shape)
        x = x.to(device)    
        z = model(x)          # [B, T, C]
        z_mean = z.mean(dim=1)  # [B, C]
        all_embeddings.append(z_mean)

embeddings_full = torch.cat(all_embeddings).cpu().numpy()

In [162]:
len(embeddings_full), len(data) # 4219

(4160, 4219)

In [163]:
val_df = pd.DataFrame(
    embeddings_full,
    index=data.index[window_size - 1:]
)

In [164]:
# val_df["Close"] = all_data["Close"].iloc[window_size-1:].values

In [165]:
val_df.head()

,0,1,2,3,4,5,6,7,8,9,...,118,119,120,121,122,123,124,125,126,127
Date,,,,,,,,,,,,,,,,,,,,,
2008-09-30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000035,0.0,0.000000,...,0.215032,0.000000,0.0,0.0,0.104896,0.0,0.0,0.071229,0.0,0.040119
2008-10-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000174,0.0,0.000000,...,0.206248,0.000000,0.0,0.0,0.104896,0.0,0.0,0.073362,0.0,0.044781
2008-10-03,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000233,0.0,0.000124,...,0.198753,0.001280,0.0,0.0,0.104896,0.0,0.0,0.075323,0.0,0.049510
2008-10-06,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000672,0.0,0.000035,...,0.191434,0.003058,0.0,0.0,0.104896,0.0,0.0,0.076693,0.0,0.056227
2008-10-07,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000484,0.0,0.000000,...,0.183625,0.004119,0.0,0.0,0.104896,0.0,0.0,0.076525,0.0,0.062015


In [166]:
def temporal_contrast_score(Z, k=20):
    pos = []
    neg = []
    
    for i in range(len(Z) - k - 1):
        pos.append(np.dot(Z[i], Z[i+1]))
        neg.append(np.dot(Z[i], Z[i+k]))
    
    return np.mean(pos) - np.mean(neg)

score = temporal_contrast_score(embeddings_full)
score

0.05380757

In [142]:
score

0.077012636